In [7]:
import os
os.chdir("D:\\Personal Projects\\Social_Ecological_Info_Cooperation\\Code\\My_Simulations")


In [8]:
os.getcwd()

'D:\\Personal Projects\\Social_Ecological_Info_Cooperation\\Code\\My_Simulations'

In [9]:
from imports import *
from helper_functions import *
from base_ecopg import *

np.set_printoptions(precision=8, suppress=True, linewidth=200)

all_information_modes = ["complete", "social", "ecological", "none"]



In [10]:
#creating strategy sets for both players for all information conditions

def add_degraded_state_policies_both_state_and_action(strategy):
    strategy_propserous_and_degraded_state = strategy.copy()

    for i in [0, 2, 4, 6]:
        x = 0
        strategy_propserous_and_degraded_state.insert(i, x)  
         #helper function that adds degraded policies even though we only explictly write strategies in the propsperous state. 
         # Here we assume that there is no cooperation in the degraded state, so we add 0s in the appropriate places.
    return strategy_propserous_and_degraded_state


def add_degraded_state_policies_only_state(strategy):
    strategy_propserous_and_degraded_state = strategy.copy()
    x = 0
    strategy_propserous_and_degraded_state.insert(0, x)

    return strategy_propserous_and_degraded_state


def create_determinstic_strategies_set_for_both_players(mode):
    """Creates deterministic strategy sets for the given information condition."""

    mae_ecopg_for_evaluating_no_of_states = create_mae_ecopg_for_given_mode_POstratAC(mode)
    number_of_states = mae_ecopg_for_evaluating_no_of_states.Q

    if mode in ["none", "social"]:
        determinstic_strategy_itertools = itertools.product([1, 0], repeat=number_of_states)
        determinisic_strategy_lists = [list(strat) for strat in determinstic_strategy_itertools]
        all_determinstic_strategy_dictionary_full = {
            str(np.round(strat)): strat for strat in determinisic_strategy_lists
        }
    else:
        number_of_prosperous_states = int(number_of_states / 2)
        determinstic_strategy_itertools = itertools.product([1, 0], repeat=number_of_prosperous_states)
        determinisic_strategy_lists = [list(strat) for strat in determinstic_strategy_itertools]
        all_determinstic_strategy_dictionary_only_prosperous = {
            str(np.round(strat)): strat for strat in determinisic_strategy_lists
        }

        if mode == "complete":
            all_determinstic_strategy_dictionary_full = {
                key: add_degraded_state_policies_both_state_and_action(value)
                for key, value in all_determinstic_strategy_dictionary_only_prosperous.items()
            }
        elif mode == "ecological":
            all_determinstic_strategy_dictionary_full = {
                key: add_degraded_state_policies_only_state(value)
                for key, value in all_determinstic_strategy_dictionary_only_prosperous.items()
            }

    strategy_set_p1 = all_determinstic_strategy_dictionary_full
    strategy_set_p2 = all_determinstic_strategy_dictionary_full

    return strategy_set_p1, strategy_set_p2


def create_policy_from_strategy(strategy_p1, strategy_p2):
    agent_1_strategy = [[x, 1 - x] for x in strategy_p1]
    agent_2_strategy = [[x, 1 - x] for x in strategy_p2]
    return np.array([agent_1_strategy, agent_2_strategy])



In [28]:
mae = create_mae_ecopg_for_given_mode_POstratAC_expanded("complete")

f = mae.env.baseenv.f[0]
c = mae.env.baseenv.c[0]
m = -6.5
p_c = q_c = np.round(mae.env.baseenv.qc[0],5)
p_r = q_r = np.round(mae.env.baseenv.qr[0],7)

discount = np.round(mae.gamma[0],2)

print(mae.env.baseenv.f)
print(mae.env.baseenv.c)

R  = f*c - c
S = f*c/2 -c
T = f*c/2
P = 0

print(f"R: {R}, S: {S}, T: {T}, P: {P}")
print(f"m = {m}, qc = {q_c:.5f}, qr = {q_r:.7f}, discount = {discount:.2f}")



[1.2 1.2]
[5 5]
R: 1.0, S: -2.0, T: 3.0, P: 0
m = -6.5, qc = 0.02000, qr = 0.0001000, discount = 0.98


first we consider the case for no-information
Here, we consider that the agents are playing strategy (D,D) as their strategy. The single step deviation would be cooperating for one round and theen subsequently defecting.
We just write this manually first. the probabilyt/payoff calculation for now

In [29]:
def V_P(m, p_c, p_r, u, gamma):
    numerator = m * p_c * (1 + gamma * p_r) + u * (1 - p_c) * (1 - gamma * (1 - p_r))
    denominator = (1 - gamma) * (1 - gamma * (1 - p_c) + gamma * p_r)
    return numerator / denominator

In [30]:
p_c_DD = p_c #if both play DD, then p_c is equal to the maximum (which is the default value of p_c)
p_r_DD = 0 #if both play D, D then there is no recovery
u_DD = 0 #if both play D, D then the payoff is 'P' of teh payoff matrix, whihc s0

V_P_dd = V_P(m, p_c_DD, p_r_DD, u_DD, discount)
print(f"V_P for (D,D) strategy: {V_P_dd:.3f}")


V_P for (D,D) strategy: -164.141


In [31]:
def V_D(V_P, gamma, p_r, m):
    numerator = V_P * gamma * p_r + m
    denominator = gamma * p_r - gamma + 1
    return numerator / denominator


V_D_dd = V_D(V_P_dd, discount, p_r_DD, m)
print(f"V_D for (D,D) strategy: {V_D_dd:.3f}")



V_D for (D,D) strategy: -324.999


In [32]:
#check if bellman funcction is valid

def check_bellman(V_P, V_D, gamma, p_c, u, m, tol=1e-9):
    lhs = V_P
    rhs = (1 - p_c)*(u + gamma*V_P) + p_c*(m + gamma*V_D)
    return abs(lhs - rhs) < tol


is_bellman_valid_DD = check_bellman(V_P_dd, V_D_dd, discount, p_c_DD, u_DD, m)
print(f"Is the Bellman equation valid for (D,D) strategy? {is_bellman_valid_DD}")

Is the Bellman equation valid for (D,D) strategy? True


In [33]:
def V_P_next_no_information(V_P_og):

    V_P_next = V_P_og 
    return V_P_next


def V_D_next_no_information(V_D_og):
    V_D_next = V_D_og 
    return V_D_next

V_P_next_dd = V_P_next_no_information(V_P_dd)
V_D_next_dd = V_D_next_no_information(V_D_dd)


def V_P_deviation(m, p_c_deviation, p_r_deviation, u_deviation, gamma, V_P_next, V_D_next):
    V_P_dev =   (1 - p_c_deviation)*(u_deviation + gamma*V_P_next) + p_c_deviation*(m + gamma*V_D_next)
    return V_P_dev

def V_D_deviation(m, p_c_deviation, p_r_deviation, gamma, V_P_next):
    numerator = V_P_next * gamma * p_r_deviation + m
    denominator = gamma * p_r_deviation - gamma + 1
    V_D_dev = numerator / denominator
    return V_D_dev


p_c_deviation_c_from_dd = p_c/2
u_deviation_c_from_dd = S #sucker's payoff when agent 1 deviates to cooperation while agent 2 plays D

V_P_deviation_c_from_dd = V_P_deviation(m, p_c_deviation_c_from_dd, p_r_DD, u_deviation_c_from_dd, discount, V_P_next_dd, V_D_next_dd)
print(f"V_P for deviation to cooperation from (D,D) strategy: {V_P_deviation_c_from_dd:.3f}")

#check if deviation is profitable
is_deviation_c_profitable_from_dd = V_P_deviation_c_from_dd > V_P_dd
print(f"Is deviation to cooperation profitable from (D,D) strategy? {is_deviation_c_profitable_from_dd}")

V_P for deviation to cooperation from (D,D) strategy: -164.480
Is deviation to cooperation profitable from (D,D) strategy? False


In [34]:
#now let us repeat the same thing when both play C, C and the deviation is to defecting for one round and then cooperating again

p_c_CC = 0 #if both play C, C then there is no chance of degradation
u_CC = R #if both play C, C then they both are rewarded for cooperation
p_r_CC = q_r #if both play C, C then the recovery probability is the default value of recovery probability

V_P_CC = V_P(m, p_c_CC, p_r_CC, u_CC, discount)
V_D_CC = V_D(V_P_CC, discount, p_r_CC, m)
print(f"V_P for (C,C) strategy: {V_P_CC:.3f}")
print(f"V_D for (C,C) strategy: {V_D_CC:.3f}")

print(f"Is the Bellman equation valid for (C,C) strategy? {check_bellman(V_P_CC, V_D_CC, discount, p_c_CC, u_CC, m)}")

V_P_next_CC = V_P_next_no_information(V_P_CC)
V_D_next_CC = V_D_next_no_information(V_D_CC)

p_c_deviation_d_from_cc = p_c/2 #if one deviates to D from C, C then the degradation probability is half of degradation probabiliti
u_deviation_d_from_cc = T #temptation payoff when agent 1 deviates to defection while agent 2 plays C
print(f"p_c_deviation_d_from_cc: {p_c_deviation_d_from_cc:.5f}, u_deviation_d_from_cc: {u_deviation_d_from_cc:.5f}, q_r: {q_r:.7f}")
V_P_deviation_d_from_cc = V_P_deviation(m, p_c_deviation_d_from_cc, p_r_CC, u_deviation_d_from_cc, discount, V_P_next_CC, V_D_next_CC)
print(f"V_P for deviation to defection from (C,C) strategy: {V_P_deviation_d_from_cc:.3f}")

is_deviation_d_profitable_from_cc = V_P_deviation_d_from_cc > V_P_CC
print(f"Is deviation to defection profitable from (C,C) strategy? {is_deviation_d_profitable_from_cc}")

V_P for (C,C) strategy: 50.000
V_D for (C,C) strategy: -323.171
Is the Bellman equation valid for (C,C) strategy? True
p_c_deviation_d_from_cc: 0.01000, u_deviation_d_from_cc: 3.00000, q_r: 0.0001000
V_P for deviation to defection from (C,C) strategy: 48.248
Is deviation to defection profitable from (C,C) strategy? False


In [ ]:
'''testing why deviation to defection from (C,C) is not profitable'''

#now let us repeat the same thing when both play C, C and the deviation is to defecting for one round and then cooperating again

p_c_CC = 0 #if both play C, C then there is no chance of degradation
u_CC = R #if both play C, C then they both are rewarded for cooperation
p_r_CC = q_r #if both play C, C then the recovery probability is the default value of recovery probability

discount_test = discount


V_no = V_P(m, p_c_CC, p_r_CC, u_CC, discount_test).
#Let us see if we can write the value function for 'no information' case, without 'prosperous' and 'degraded' labels, and whether that makes any difference
V_no = V_P_next_no_information(V_no)

p_c_deviation_d_from_cc = p_c/2 #if one deviates to D from C, C then the degradation probability is half of degradation probabiliti
u_deviation_d_from_cc = T #temptation payoff when agent 1 deviates to defection while agent 2 plays C
print(f"p_c_deviation_d_from_cc: {p_c_deviation_d_from_cc:.5f}, u_deviation_d_from_cc: {u_deviation_d_from_cc:.5f}, q_r: {q_r:.7f}")
V_P_deviation_d_from_cc = V_P_deviation(m, p_c_deviation_d_from_cc, p_r_CC, u_deviation_d_from_cc, discount_test, V_P_next_CC, V_D_next_CC)
print(f"V_P for deviation to defection from (C,C) strategy: {V_P_deviation_d_from_cc:.3f}")

is_deviation_d_profitable_from_cc = (V_P_deviation_d_from_cc > V_P_CC)
print(f"Is deviation to defection profitable from (C,C) strategy? {is_deviation_d_profitable_from_cc}")

V_P for (C,C) strategy: 40.000
V_D for (C,C) strategy: -258.835
p_c_deviation_d_from_cc: 0.01000, u_deviation_d_from_cc: 3.00000, q_r: 0.0001000
V_P for deviation to defection from (C,C) strategy: 38.991
Is deviation to defection profitable from (C,C) strategy? False


In [ ]:
#For C,C, let us check if deviation is possible in the degraded state and if it is profitable, 
#this means checking if C,C 

In [ ]:

def check_spne(strategy_p1, strategy_p2, information_condition):
    """Placeholder SPNE check.

    Intended logic later:
    1. Evaluate cumulative reward when both agents follow the supplied strategies.
    2. Check whether a one-step deviation followed by returning gives a higher reward.
    3. If any one-shot deviation is better, return not SPNE; otherwise return SPNE.
    """

    policy = create_policy_from_strategy(strategy_p1, strategy_p2)
    mae = create_mae_ecopg_for_given_mode_POstratAC(information_condition)

    agent_1_cumulative_reward = mae


    # result = {
    #     "information_condition": information_condition,
    #     "strategy_p1": list(strategy_p1),
    #     "strategy_p2": list(strategy_p2),
    #     "policy": policy,
    #     "mae": mae,
    #     "baseline_cumulative_reward": None,
    #     "best_one_step_deviation_reward": None,
    #     "is_spne": None,
    #     "notes": "Placeholder only. Add reward and one-step deviation logic here.",
    # }

    return result


In [ ]:
# Manual test before running the full loop
manual_information_condition = "social"
manual_strategy_p1 = [0, 0, 0, 0]
manual_strategy_p2 = [0, 0, 0, 0]

manual_spne_check = check_spne(
    strategy_p1=manual_strategy_p1,
    strategy_p2=manual_strategy_p2,
    information_condition=manual_information_condition,
)

manual_spne_check


In [ ]:
all_spne_checks = []

for information_condition in all_information_modes:
    strategy_set_p1, strategy_set_p2 = create_determinstic_strategies_set_for_both_players(information_condition)

    for strategy_p1, strategy_p2 in itertools.product(strategy_set_p1.values(), strategy_set_p2.values()):
        spne_result = check_spne(
            strategy_p1=strategy_p1,
            strategy_p2=strategy_p2,
            information_condition=information_condition,
        )
        all_spne_checks.append(spne_result)

len(all_spne_checks)


In [ ]:
spne_results_df = pd.DataFrame(
    [
        {
            "information_condition": result["information_condition"],
            "strategy_p1": result["strategy_p1"],
            "strategy_p2": result["strategy_p2"],
            "baseline_cumulative_reward": result["baseline_cumulative_reward"],
            "best_one_step_deviation_reward": result["best_one_step_deviation_reward"],
            "is_spne": result["is_spne"],
            "notes": result["notes"],
        }
        for result in all_spne_checks
    ]
)

spne_results_df.head()
